# Дедупликация merged_2023_2025.csv и сравнение с crm_last_version.csv

Тетрадка работает с уже готовым `merged_2023_2025.csv` (повторно объединять 6 файлов не нужно).

Логика дедупликации повторяет `analysis_crm_segments.ipynb` и `analysis_scenario_ipu_results.ipynb`.

In [ ]:
import os
import pandas as pd

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
MERGED_FILE = f"{DATA_DIR}/merged_2023_2025.csv"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"

DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO = pd.Timestamp("2025-12-31")
ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]
DEDUP_KEY = ["inn", "fp_num", "fp_type", "IDENTIFICATION_DATE"]

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}

print("DATA_DIR:", DATA_DIR)
print("MERGED_FILE:", MERGED_FILE)
print("CRM_FILE:", CRM_FILE)

In [ ]:
def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


def prepare_and_dedup(df_raw, label="dataset"):
    df = df_raw.copy()
    steps = []

    def add_step(step_name, rows_after, rows_before=None):
        removed = None if rows_before is None else (rows_before - rows_after)
        steps.append({"step": step_name, "rows_after": rows_after, "removed": removed})

    add_step("start", len(df))

    if "VAL" in df.columns:
        before = len(df)
        df = df[df["VAL"].astype(str).str.strip().isin(ALLOWED_SOURCES)].copy()
        add_step("filter_VAL", len(df), before)

    if "X_INN" not in df.columns:
        raise KeyError(f"{label}: отсутствует колонка X_INN")
    df["inn"] = df["X_INN"].apply(normalize_inn)

    if "IDENTIFICATION_DATE" not in df.columns:
        raise KeyError(f"{label}: отсутствует колонка IDENTIFICATION_DATE")
    df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")

    before = len(df)
    df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()
    add_step("filter_period", len(df), before)

    if "X_AREA_RESP" in df.columns:
        df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
        df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)
        before = len(df)
        df = df[df["segment"].notna()].copy()
        add_step("map_segment", len(df), before)

    if "NUMBER_FP_SFP" not in df.columns:
        raise KeyError(f"{label}: отсутствует колонка NUMBER_FP_SFP")
    df["fp_num"] = df["NUMBER_FP_SFP"].astype(str).str.strip()

    if "TYPE_FP" in df.columns:
        df["fp_type"] = df["TYPE_FP"].astype(str).str.strip().replace({"FP": "ФП", "SFP": "СФП"})
    else:
        df["fp_type"] = None

    before = len(df)
    df = df.dropna(subset=["inn", "NUMBER_FP_SFP"]).copy()
    add_step("dropna_inn_fp", len(df), before)

    before = len(df)
    dedup = df.drop_duplicates(subset=DEDUP_KEY).copy()
    add_step("dedup_by_key", len(dedup), before)

    metrics = {
        "dataset": label,
        "rows_raw": len(df_raw),
        "rows_after_dedup": len(dedup),
        "unique_inn_after_dedup": dedup["inn"].nunique(),
        "unique_cards_after_dedup": dedup[DEDUP_KEY].drop_duplicates().shape[0],
        "columns": dedup.shape[1],
    }

    return dedup, pd.DataFrame(steps), metrics

In [ ]:
if not os.path.exists(MERGED_FILE):
    raise FileNotFoundError(f"Не найден файл: {MERGED_FILE}")

merged_raw = pd.read_csv(MERGED_FILE, dtype=str, low_memory=False)
merged_dedup, merged_steps, merged_metrics = prepare_and_dedup(merged_raw, label="merged_2023_2025")

print("=== merged_2023_2025.csv ===")
display(pd.DataFrame([merged_metrics]))
display(merged_steps)

print("Сохранить дедуплицированную выгрузку?")
DEDUP_FILE = f"{DATA_DIR}/merged_2023_2025_dedup.csv"
merged_dedup.to_csv(DEDUP_FILE, index=False, encoding="utf-8")
print("Сохранено:", DEDUP_FILE)

In [ ]:
# Сравнение с crm_last_version.csv по той же логике (если файл присутствует)
if os.path.exists(CRM_FILE):
    crm_raw = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
    crm_dedup, crm_steps, crm_metrics = prepare_and_dedup(crm_raw, label="crm_last_version")

    comp = pd.DataFrame([merged_metrics, crm_metrics])
    print("=== Сравнение merged_2023_2025 vs crm_last_version ===")
    display(comp)

    compare_pairs = [
        ("rows_after_dedup", "Строк после дедупликации"),
        ("unique_inn_after_dedup", "Уникальных ИНН"),
        ("unique_cards_after_dedup", "Уникальных карточек по ключу"),
    ]
    for key, title in compare_pairs:
        m = merged_metrics[key]
        c = crm_metrics[key]
        print(f"{title}: merged={m:,}, crm={c:,}, delta={m-c:+,}")
else:
    print(f"Файл не найден, сравнение пропущено: {CRM_FILE}")